# Training

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Load your data
df = pd.read_pickle("../data/processed/preprocessed_weather.pkl")

# --- Preprocessing ---

# Convert time to useful numeric features
df["time"] = pd.to_datetime(df["time"])
df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month
df["day"] = df["time"].dt.day

# Drop original time column
df = df.drop(columns=["time"])

# Convert categorical to numeric (one-hot encoding)
df = pd.get_dummies(df, columns=["weather_code"], drop_first=True)

df.head()

,temperature_2m_max,temperature_2m_min,temperature_2m_mean,apparent_temperature_max,apparent_temperature_min,relative_humidity_2m_max,relative_humidity_2m_min,relative_humidity_2m_mean,wind_speed_10m_max,wind_gusts_10m_max,...,day,weather_code_1,weather_code_2,weather_code_3,weather_code_51,weather_code_53,weather_code_55,weather_code_61,weather_code_63,weather_code_65
0,21.8,6.5,14.3,20.9,4.3,94,39,68,4.9,11.4,...,1,False,False,False,False,False,False,False,False,False
1,22.6,8.0,14.6,21.7,6.0,95,41,71,6.4,9.8,...,2,False,False,False,False,False,False,False,False,False
2,22.6,7.8,14.6,21.2,6.2,97,41,73,6.5,13.4,...,3,False,False,False,False,False,False,False,False,False
3,22.2,8.0,14.7,21.6,6.1,96,35,69,5.8,10.3,...,4,False,False,True,False,False,False,False,False,False
4,23.4,9.9,16.0,21.5,8.2,86,42,66,7.9,16.6,...,5,False,False,True,False,False,False,False,False,False


## Linear Regression

In [8]:
# Set up training loop with Weights and Biases

from sklearn.linear_model import LinearRegression, Ridge, Lasso
import wandb

training_params = [
    # --- LinearRegression: only knob worth turning is fit_intercept ---
    {
        "run_name": "linear_reg_no_intercept",
        "model_class": LinearRegression,
        "params": {"fit_intercept": False}
    },
    {
        "run_name": "linear_reg_with_intercept",
        "model_class": LinearRegression,
        "params": {"fit_intercept": True}
    },

    # --- Ridge: L2 regularization, sweep alpha (regularization strength) ---
    {
        "run_name": "ridge_alpha_0.1",
        "model_class": Ridge,
        "params": {"alpha": 0.1, "fit_intercept": True}
    },
    {
        "run_name": "ridge_alpha_1.0",
        "model_class": Ridge,
        "params": {"alpha": 1.0, "fit_intercept": True}
    },
    {
        "run_name": "ridge_alpha_10.0",
        "model_class": Ridge,
        "params": {"alpha": 10.0, "fit_intercept": True}
    },

    # --- Lasso: L1 regularization, also sweeps alpha ---
    {
        "run_name": "lasso_alpha_0.1",
        "model_class": Lasso,
        "params": {"alpha": 0.1, "fit_intercept": True, "max_iter": 5000}
    },
    {
        "run_name": "lasso_alpha_1.0",
        "model_class": Lasso,
        "params": {"alpha": 1.0, "fit_intercept": True, "max_iter": 5000}
    },
]

# Prepare data
X = df.drop(columns=["apparent_temperature_max"], axis=1)
y = df["apparent_temperature_max"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Run each of the params with Weights and Biases
for config in training_params:
    run_name = config["run_name"]
    params = config["params"]
    model_class = config["model_class"]

    with wandb.init(project="weather-prediction", name=run_name, config=params) as run:
        model = model_class(**params)   # <-- use model_class here
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        run.log({"mse": mse, "r2": r2})

mse,▁
r2,▁
mse,0.45607
r2,0.99224


mse,▁
r2,▁
mse,0.45953
r2,0.99219


mse,▁
r2,▁
mse,0.45962
r2,0.99218


mse,▁
r2,▁
mse,0.4607
r2,0.99217


mse,▁
r2,▁
mse,0.47092
r2,0.99199


mse,▁
r2,▁
mse,0.49508
r2,0.99158


mse,▁
r2,▁
mse,0.94337
r2,0.98396


## KNN Regressor

In [9]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import wandb

knn_training_params = [
    # --- Sweep n_neighbors with uniform weighting ---
    {"run_name": "knn_k3_uniform", "params": {"n_neighbors": 3, "weights": "uniform", "metric": "euclidean", "algorithm": "auto"}},
    {"run_name": "knn_k5_uniform", "params": {"n_neighbors": 5, "weights": "uniform", "metric": "euclidean", "algorithm": "auto"}},
    {"run_name": "knn_k10_uniform", "params": {"n_neighbors": 10, "weights": "uniform", "metric": "euclidean", "algorithm": "auto"}},
    {"run_name": "knn_k20_uniform", "params": {"n_neighbors": 20, "weights": "uniform", "metric": "euclidean", "algorithm": "auto"}},
    {"run_name": "knn_k50_uniform", "params": {"n_neighbors": 50, "weights": "uniform", "metric": "euclidean", "algorithm": "auto"}},

    # --- Sweep n_neighbors with distance weighting ---
    {"run_name": "knn_k3_distance", "params": {"n_neighbors": 3, "weights": "distance", "metric": "euclidean", "algorithm": "auto"}},
    {"run_name": "knn_k5_distance", "params": {"n_neighbors": 5, "weights": "distance", "metric": "euclidean", "algorithm": "auto"}},
    {"run_name": "knn_k10_distance", "params": {"n_neighbors": 10, "weights": "distance", "metric": "euclidean", "algorithm": "auto"}},
    {"run_name": "knn_k20_distance", "params": {"n_neighbors": 20, "weights": "distance", "metric": "euclidean", "algorithm": "auto"}},
    {"run_name": "knn_k50_distance", "params": {"n_neighbors": 50, "weights": "distance", "metric": "euclidean", "algorithm": "auto"}},

    # --- Sweep distance metrics ---
    {"run_name": "knn_k10_manhattan", "params": {"n_neighbors": 10, "weights": "distance", "metric": "manhattan", "algorithm": "auto"}},
    {"run_name": "knn_k10_chebyshev", "params": {"n_neighbors": 10, "weights": "distance", "metric": "chebyshev", "algorithm": "auto"}},
    {"run_name": "knn_k10_minkowski_p3", "params": {"n_neighbors": 10, "weights": "distance", "metric": "minkowski", "metric_params": {"p": 3}, "algorithm": "auto"}},

    # --- Sweep algorithms ---
    {"run_name": "knn_k10_balltree", "params": {"n_neighbors": 10, "weights": "distance", "metric": "euclidean", "algorithm": "ball_tree"}},
    {"run_name": "knn_k10_kdtree", "params": {"n_neighbors": 10, "weights": "distance", "metric": "euclidean", "algorithm": "kd_tree"}},
]

# KNN-specific training loop (with scaling pipeline)
for config in knn_training_params:
    run_name = config["run_name"]
    params = config["params"]

    with wandb.init(project="weather-prediction", name=run_name, config=params) as run:

        # Build pipeline — scaler is essential for KNN
        knn_model = KNeighborsRegressor(**params)
        
        pipeline = Pipeline([
            ("scaler", StandardScaler()),
            ("model", knn_model)
        ])

        pipeline.fit(X_train, y_train)

        y_pred = pipeline.predict(X_test)
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        run.log({"mse": mse, "r2": r2})

mse,▁
r2,▁
mse,4.34275
r2,0.92615


mse,▁
r2,▁
mse,4.19748
r2,0.92862


mse,▁
r2,▁
mse,4.32482
r2,0.92645


mse,▁
r2,▁
mse,5.07167
r2,0.91375


mse,▁
r2,▁
mse,6.04382
r2,0.89722


...

mse,▁
r2,▁
mse,10.82586
r2,0.8159


/Users/nathanhwang/Coding/WeatherML/.venv/lib/python3.12/site-packages/sklearn/neighbors/_regression.py:227: SyntaxWarning: Parameter p is found in metric_params. The corresponding parameter from __init__ is ignored.
  return self._fit(X, y)


mse,▁
r2,▁
mse,5.22582
r2,0.91113


mse,▁
r2,▁
mse,3.95874
r2,0.93268


mse,▁
r2,▁
mse,3.95874
r2,0.93268


# Polynomial Regression

In [10]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import wandb

import sys
import os

sys.path.append(os.path.abspath(".."))

poly_training_params = [
    # --- Degree sweep with fixed alpha ---
    {"run_name": "poly_deg1_ridge1.0",  "params": {"degree": 1, "alpha": 1.0}},
    {"run_name": "poly_deg2_ridge1.0",  "params": {"degree": 2, "alpha": 1.0}},
    {"run_name": "poly_deg3_ridge1.0",  "params": {"degree": 3, "alpha": 1.0}},

    # --- Alpha sweep at degree 2 ---
    {"run_name": "poly_deg2_ridge0.01",  "params": {"degree": 2, "alpha": 0.01}},
    {"run_name": "poly_deg2_ridge0.1",   "params": {"degree": 2, "alpha": 0.1}},
    {"run_name": "poly_deg2_ridge10.0",  "params": {"degree": 2, "alpha": 10.0}},
    {"run_name": "poly_deg2_ridge100.0", "params": {"degree": 2, "alpha": 100.0}},

    # --- Alpha sweep at degree 3 ---
    {"run_name": "poly_deg3_ridge0.1",   "params": {"degree": 3, "alpha": 0.1}},
    {"run_name": "poly_deg3_ridge10.0",  "params": {"degree": 3, "alpha": 10.0}},
    {"run_name": "poly_deg3_ridge100.0", "params": {"degree": 3, "alpha": 100.0}},

    # --- interaction_only: skip x² terms, keep only cross-features ---
    {"run_name": "poly_deg2_interactions_only", "params": {"degree": 2, "alpha": 1.0, "interaction_only": True}},
    {"run_name": "poly_deg3_interactions_only", "params": {"degree": 3, "alpha": 1.0, "interaction_only": True}},
]

for config in poly_training_params:
    run_name = config["run_name"]
    params   = config["params"]

    # Separate pipeline params from W&B config
    degree           = params["degree"]
    alpha            = params["alpha"]
    interaction_only = params.get("interaction_only", False)

    with wandb.init(project="weather-prediction", name=run_name, config=params) as run:
        pipeline = Pipeline([
            ("poly",   PolynomialFeatures(
                            degree=degree,
                            include_bias=False,
                            interaction_only=interaction_only
                       )),
            ("scaler", StandardScaler()),
            ("model",  Ridge(alpha=alpha, fit_intercept=True)),
        ])
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)

        mse  = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2   = r2_score(y_test, y_pred)

        # Log feature count so W&B lets you see expansion cost
        n_features_out = pipeline.named_steps["poly"].n_output_features_

        run.log({
            "mse":            mse,
            "rmse":           rmse,
            "r2":             r2,
            "n_features_out": n_features_out,
        })

mse,▁
n_features_out,▁
r2,▁
rmse,▁
mse,0.45955
n_features_out,29
r2,0.99218
rmse,0.6779


mse,▁
n_features_out,▁
r2,▁
rmse,▁
mse,0.39336
n_features_out,464
r2,0.99331
rmse,0.62719


mse,▁
n_features_out,▁
r2,▁
rmse,▁
mse,0.48852
n_features_out,4959
r2,0.99169
rmse,0.69894


mse,▁
n_features_out,▁
r2,▁
rmse,▁
mse,0.43039
n_features_out,464
r2,0.99268
rmse,0.65604


mse,▁
n_features_out,▁
r2,▁
rmse,▁
mse,0.41454
n_features_out,464
r2,0.99295
rmse,0.64385


...

mse,▁
n_features_out,▁
r2,▁
rmse,▁
mse,0.68334
n_features_out,4959
r2,0.98838
rmse,0.82664


mse,▁
n_features_out,▁
r2,▁
rmse,▁
mse,0.40902
n_features_out,4959
r2,0.99304
rmse,0.63954


mse,▁
n_features_out,▁
r2,▁
rmse,▁
mse,0.40909
n_features_out,4959
r2,0.99304
rmse,0.6396


mse,▁
n_features_out,▁
r2,▁
rmse,▁
mse,0.39742
n_features_out,435
r2,0.99324
rmse,0.63041


mse,▁
n_features_out,▁
r2,▁
rmse,▁
mse,0.50596
n_features_out,4089
r2,0.9914
rmse,0.71131


# Random Forest Regression

In [11]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import wandb

# Parameter sweep
rf_training_params = [
    # --- Sweep n_estimators ---
    {"run_name": "rf_n100_depth5",  "params": {"n_estimators": 100, "max_depth": 5,    "min_samples_split": 2,  "max_features": "sqrt"}},
    {"run_name": "rf_n200_depth5",  "params": {"n_estimators": 200, "max_depth": 5,    "min_samples_split": 2,  "max_features": "sqrt"}},
    {"run_name": "rf_n500_depth5",  "params": {"n_estimators": 500, "max_depth": 5,    "min_samples_split": 2,  "max_features": "sqrt"}},

    # --- Sweep max_depth ---
    {"run_name": "rf_n200_depth3",  "params": {"n_estimators": 200, "max_depth": 3,    "min_samples_split": 2,  "max_features": "sqrt"}},
    {"run_name": "rf_n200_depthNone", "params": {"n_estimators": 200, "max_depth": None, "min_samples_split": 2, "max_features": "sqrt"}},
    {"run_name": "rf_n200_depth10", "params": {"n_estimators": 200, "max_depth": 10,   "min_samples_split": 2,  "max_features": "sqrt"}},

    # --- Sweep min_samples_split ---
    {"run_name": "rf_n200_split5",  "params": {"n_estimators": 200, "max_depth": 5,    "min_samples_split": 5,  "max_features": "sqrt"}},
    {"run_name": "rf_n200_split10", "params": {"n_estimators": 200, "max_depth": 5,    "min_samples_split": 10, "max_features": "sqrt"}},

    # --- Sweep max_features ---
    {"run_name": "rf_n200_feat_log2",  "params": {"n_estimators": 200, "max_depth": 5, "min_samples_split": 2, "max_features": "log2"}},
    {"run_name": "rf_n200_feat_none",  "params": {"n_estimators": 200, "max_depth": 5, "min_samples_split": 2, "max_features": 1.0}},
]

for config in rf_training_params:
    run_name = config["run_name"]
    params   = config["params"]

    with wandb.init(project="weather-prediction", name=run_name, config=params) as run:
        model = RandomForestRegressor(
            **params,
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        mse  = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2   = r2_score(y_test, y_pred)

        # Feature importances — useful to inspect in W&B
        importances = dict(zip(X.columns, model.feature_importances_.round(4)))

        run.log({
            "mse":              mse,
            "rmse":             rmse,
            "r2":               r2,
            "feature_importances": importances,
        })

mse,▁
r2,▁
rmse,▁
mse,1.74696
r2,0.97029
rmse,1.32173


mse,▁
r2,▁
rmse,▁
mse,1.75652
r2,0.97013
rmse,1.32534


mse,▁
r2,▁
rmse,▁
mse,1.71027
r2,0.97092
rmse,1.30777


mse,▁
r2,▁
rmse,▁
mse,3.67261
r2,0.93754
rmse,1.9164


mse,▁
r2,▁
rmse,▁
mse,0.90545
r2,0.9846
rmse,0.95155


mse,▁
r2,▁
rmse,▁
mse,0.97247
r2,0.98346
rmse,0.98614


mse,▁
r2,▁
rmse,▁
mse,1.77522
r2,0.96981
rmse,1.33237


mse,▁
r2,▁
rmse,▁
mse,1.76583
r2,0.96997
rmse,1.32885


mse,▁
r2,▁
rmse,▁
mse,2.20124
r2,0.96257
rmse,1.48366


mse,▁
r2,▁
rmse,▁
mse,1.31117
r2,0.9777
rmse,1.14506


# Gradient Boosting

In [13]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import wandb

# Prepare data
df_clean = df.dropna(subset=["apparent_temperature_max"])

# Parameter sweep
lgb_training_params = [
    # --- Baseline ---
    {"run_name": "lgb_baseline",         "params": {"n_estimators": 200, "learning_rate": 0.1,  "max_depth": -1,  "num_leaves": 31,  "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 1.0,  "reg_alpha": 0.0, "reg_lambda": 0.0}},

    # --- Sweep learning rate ---
    {"run_name": "lgb_lr0.01",           "params": {"n_estimators": 200, "learning_rate": 0.01, "max_depth": -1,  "num_leaves": 31,  "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 1.0,  "reg_alpha": 0.0, "reg_lambda": 0.0}},
    {"run_name": "lgb_lr0.05",           "params": {"n_estimators": 200, "learning_rate": 0.05, "max_depth": -1,  "num_leaves": 31,  "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 1.0,  "reg_alpha": 0.0, "reg_lambda": 0.0}},
    {"run_name": "lgb_lr0.3",            "params": {"n_estimators": 200, "learning_rate": 0.3,  "max_depth": -1,  "num_leaves": 31,  "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 1.0,  "reg_alpha": 0.0, "reg_lambda": 0.0}},

    # --- Sweep num_leaves (controls tree complexity) ---
    {"run_name": "lgb_leaves15",         "params": {"n_estimators": 200, "learning_rate": 0.1,  "max_depth": -1,  "num_leaves": 15,  "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 1.0,  "reg_alpha": 0.0, "reg_lambda": 0.0}},
    {"run_name": "lgb_leaves63",         "params": {"n_estimators": 200, "learning_rate": 0.1,  "max_depth": -1,  "num_leaves": 63,  "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 1.0,  "reg_alpha": 0.0, "reg_lambda": 0.0}},
    {"run_name": "lgb_leaves127",        "params": {"n_estimators": 200, "learning_rate": 0.1,  "max_depth": -1,  "num_leaves": 127, "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 1.0,  "reg_alpha": 0.0, "reg_lambda": 0.0}},

    # --- Sweep n_estimators ---
    {"run_name": "lgb_n100",             "params": {"n_estimators": 100, "learning_rate": 0.1,  "max_depth": -1,  "num_leaves": 31,  "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 1.0,  "reg_alpha": 0.0, "reg_lambda": 0.0}},
    {"run_name": "lgb_n500",             "params": {"n_estimators": 500, "learning_rate": 0.1,  "max_depth": -1,  "num_leaves": 31,  "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 1.0,  "reg_alpha": 0.0, "reg_lambda": 0.0}},
    {"run_name": "lgb_n1000",            "params": {"n_estimators": 1000,"learning_rate": 0.05, "max_depth": -1,  "num_leaves": 31,  "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 1.0,  "reg_alpha": 0.0, "reg_lambda": 0.0}},

    # --- Sweep subsampling (reduces overfitting) ---
    {"run_name": "lgb_subsample0.7",     "params": {"n_estimators": 200, "learning_rate": 0.1,  "max_depth": -1,  "num_leaves": 31,  "min_child_samples": 20, "subsample": 0.7,  "colsample_bytree": 1.0,  "reg_alpha": 0.0, "reg_lambda": 0.0}},
    {"run_name": "lgb_colsample0.7",     "params": {"n_estimators": 200, "learning_rate": 0.1,  "max_depth": -1,  "num_leaves": 31,  "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 0.7,  "reg_alpha": 0.0, "reg_lambda": 0.0}},
    {"run_name": "lgb_sub0.7_col0.7",   "params": {"n_estimators": 200, "learning_rate": 0.1,  "max_depth": -1,  "num_leaves": 31,  "min_child_samples": 20, "subsample": 0.7,  "colsample_bytree": 0.7,  "reg_alpha": 0.0, "reg_lambda": 0.0}},

    # --- Sweep regularization ---
    {"run_name": "lgb_l1_0.1",          "params": {"n_estimators": 200, "learning_rate": 0.1,  "max_depth": -1,  "num_leaves": 31,  "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 1.0,  "reg_alpha": 0.1, "reg_lambda": 0.0}},
    {"run_name": "lgb_l2_0.1",          "params": {"n_estimators": 200, "learning_rate": 0.1,  "max_depth": -1,  "num_leaves": 31,  "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 1.0,  "reg_alpha": 0.0, "reg_lambda": 0.1}},
    {"run_name": "lgb_l1_l2",           "params": {"n_estimators": 200, "learning_rate": 0.1,  "max_depth": -1,  "num_leaves": 31,  "min_child_samples": 20, "subsample": 1.0,  "colsample_bytree": 1.0,  "reg_alpha": 0.1, "reg_lambda": 0.1}},
]

for config in lgb_training_params:
    run_name = config["run_name"]
    params   = config["params"]

    with wandb.init(project="weather-prediction", name=run_name, config=params) as run:
        model = lgb.LGBMRegressor(
            **params,
            random_state=42,
            n_jobs=-1,
            verbose=-1       # suppress LightGBM's per-tree console output
        )

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        mse  = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2   = r2_score(y_test, y_pred)

        importances = dict(zip(X.columns, model.feature_importances_.round(4)))

        run.log({
            "mse":               mse,
            "rmse":              rmse,
            "r2":                r2,
            "feature_importances": importances,
        })

mse,▁
r2,▁
rmse,▁
mse,0.56577
r2,0.99038
rmse,0.75218


mse,▁
r2,▁
rmse,▁
mse,2.04174
r2,0.96528
rmse,1.42889


mse,▁
r2,▁
rmse,▁
mse,0.56978
r2,0.99031
rmse,0.75484


mse,▁
r2,▁
rmse,▁
mse,0.60857
r2,0.98965
rmse,0.78011


mse,▁
r2,▁
rmse,▁
mse,0.61934
r2,0.98947
rmse,0.78698


...

mse,▁
r2,▁
rmse,▁
mse,0.61035
r2,0.98962
rmse,0.78125


mse,▁
r2,▁
rmse,▁
mse,0.61035
r2,0.98962
rmse,0.78125


mse,▁
r2,▁
rmse,▁
mse,0.58054
r2,0.99013
rmse,0.76193


mse,▁
r2,▁
rmse,▁
mse,0.57062
r2,0.9903
rmse,0.7554


mse,▁
r2,▁
rmse,▁
mse,0.57653
r2,0.9902
rmse,0.75929
